# Spontaneous Bacterial Peritonitis

**Primary**:
- Rates of antibiotic resistance

**Secondary**:
- Other stuff

**Inclusion critera**
- [ ] Culture positive SBP (report culture negative rates (~40%, usually)...include all organisms, focus on G+/- as subgroups
- [ ] $>=$ 18 years of age
- [ ] One or more measurements of PMN > 250
- [ ] Spontaneous Bacterial Peritonitis (SBP)

**Exclusion criteria**
- [ ] Patients with gram-positive, anaerobic, or fungal organisms
- [ ] Polymicrobial growth (?)
- [ ] Bacteriascites (?)
- [ ] Secondary peritonitis (e.g. perforation) since it brings in different flora

## Definitions
### MDRO (Magiorakos et al.)
- Acquired non-susceptibility to:
    - **MDR**: 1 agent in 3 or more categories
- Instrinsic resistance is excluded from category count.
- Doesn't cover **coagulase-negative**, **staphylococci**, **streptococci**, and **anaerobes** so need a prespecified rule.
- Designed for surveillance so might misalign with clinically meanful isolates.
- Need to include **ESBL, CRE, MRSA**, and **VRE** separately.

### SBP
- Ascitic fluid PMN $>=$ 250 cells/mm3 in patients with cirrhosis and ascites, with no evidence of surgically treated intra-abdominal source (AASLD, EASL both use this):
    - **Culture positive SBP**: PMN $>=$ 250 with positive ascitic culture (usually monomicrobial)
    - **Culture negative neutrocytic ascites (CNNA)**: PMN $>=$ 250 with negative culture
    - **Monomicrobial bacterascites (PMN < 250)**: not SBP by standard definition, but often included as secondary category since it can progress.


In [1]:
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
plt.style.use('ggplot')

In [3]:
engine = create_engine(f"postgresql://postgres@localhost:5432/mimiciv")

# Data extraction

In [4]:
def run_query(query):
    with engine.begin() as c:
        c.exec_driver_sql(query)

def query_table(table):
    return pd.read_sql(f'select * from {table}', engine)

In [5]:
def clear_database():
    run_query('drop schema sbp cascade;')
    run_query('create schema sbp;')

clear_database()

## SBP diagnosis based on elevated PMN count
Define SBP as PMN >= 250 cells/ml

In [6]:
# create table of pmn counts
q = """
drop table if exists sbp.pmn;
create table if not exists sbp.pmn as (
    with asc_raw as (
        select
            a.subject_id,
            a.hadm_id,
            a.charttime,
            max(case when a.itemid = 52065 then valuenum else null end) as total_nuc_cells,
            max(case when a.itemid = 51125 then valuenum else null end) as poly_pct
        from mimiciv_hosp.labevents a
        left join mimiciv_hosp.d_labitems b
        using (itemid)
        where fluid = 'Ascites'
        and hadm_id is not null
        group by subject_id, hadm_id, charttime
    )
    select
        subject_id,
        hadm_id,
        charttime,
        poly_pct * total_nuc_cells / 100 as pmn_count
    from asc_raw
    where total_nuc_cells is not null and poly_pct is not null
);
"""
run_query(q)

In [7]:
# create table of sbp episodes...14 day gap between high PMN starts new episode
q = """
drop table if exists sbp.sbp_episodes;
create table if not exists sbp.sbp_episodes as
    with sbp as (
        select
            subject_id,
            hadm_id,
            charttime,
            pmn_count,
            1 as sbp
        from sbp.pmn
        where pmn_count > 250
    ), gaps as (
        select
            subject_id,
            hadm_id,
            charttime,
            pmn_count,
            sbp,
            lag(charttime) over (partition by subject_id, hadm_id order by charttime) as prev_charttime
        from sbp
    ), flagged as (
        select
            subject_id,
            hadm_id,
            charttime,
            pmn_count,
            sbp,
            case
                when prev_charttime is null then 1
                when (charttime - prev_charttime) >= interval '14 days' then 1
                else 0
            end as is_new_episode
        from gaps
    ), episodes as (
        select
            subject_id,
            hadm_id,
            charttime,
            pmn_count,
            sbp,
            is_new_episode,
            1 - is_new_episode as is_same_episode_as_prev,
            sum(is_new_episode) over (
                partition by
                    subject_id,
                    hadm_id
                order by charttime
                rows between unbounded preceding and current row
            ) as episode_id
        from flagged
    ), tmp as (
        select
            subject_id,
            hadm_id,
            charttime,
            sbp,
            episode_id,
            case when charttime = min(charttime) over (partition by subject_id, hadm_id, episode_id) then pmn_count else null end as start_pmn,
            case when charttime = max(charttime) over (partition by subject_id, hadm_id, episode_id) then pmn_count else null end as stop_pmn            
        from episodes
    )
    select
        subject_id,
        hadm_id,
        episode_id,
        sbp,
        min(charttime) as starttime,
        max(charttime) as stoptime,
        max(start_pmn) as start_pmn,
        max(stop_pmn) as stop_pmn
    from tmp
    group by subject_id, hadm_id, episode_id, sbp
;
"""
run_query(q)

In [8]:
# create pmn index as the first sbp episode for a given admission 
q = """
drop table if exists sbp.pmn_index;
create table if not exists sbp.pmn_index as (
    select
        subject_id,
        hadm_id,
        starttime as charttime,
        start_pmn as pmn_count
    from sbp.sbp_episodes
    where episode_id = 1
);
"""
run_query(q)

## Demographics

In [9]:
# create demographic table
q = """
drop table if exists sbp.demo;
create table sbp.demo as (
    select
        subject_id,
        hadm_id,
        case when gender = 'M' then 1 else 0 end as male,
        (anchor_age + (extract(year from charttime) - anchor_year)) as age,
        admission_type,
        admission_location,
        discharge_location,
        insurance,
        language,
        marital_status,
        race
    from sbp.pmn_index
    left join mimiciv_hosp.patients
    using (subject_id)
    left join mimiciv_hosp.admissions
    using (subject_id, hadm_id)
);
"""
run_query(q)

## Communitiy acquired vs Hospital Acquired
- **Community acquired**: if SBP within first 48 hours and no prior hospital exposure
- **Hospital associated**: if SBP within first 48 hours with hospital exposure in previous 90 days
- **Hospital acquired**: if SBP after 48 hours

In [10]:
# community acquired
q = """
drop table if exists sbp.community_acquired;
create table if not exists sbp.community_acquired as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case
            when idx.charttime <= (adm1.admittime + interval '48 hours') -- if sbp within first 48 hours of admission 
            then 1
            else 0
        end) as community_acquired,
        max(case
            when idx.charttime > (adm1.admittime + interval '48 hours') -- if sbp after 48h from admission
            then 1 else 0 end) as hospital_acquired
    from sbp.pmn_index as idx
    left join mimiciv_hosp.admissions as adm1
    using (subject_id, hadm_id)
    left join mimiciv_hosp.admissions as adm2
    on idx.subject_id = adm2.subject_id
    and idx.hadm_id != adm2.hadm_id
    and adm2.dischtime between (adm1.admittime - interval '90 days') and (adm1.admittime)
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

In [11]:
# create table indicating whether SBP patients have a dx code at current visit 
q = """
drop table if exists sbp.pmn_dx;
create table if not exists sbp.pmn_dx as (
    select
        subject_id,
        hadm_id,
        max(case when icd_code in ('K652', '56723') then 1 else 0 end) as sbp_dx
    from sbp.pmn_index a
    left join mimiciv_hosp.diagnoses_icd b
    using (subject_id, hadm_id)
    group by subject_id, hadm_id
);
"""
run_query(q)

## Exclusions

Anyone with surgical procedure or other diagnosis that may result in bacterial peritonitis occurring between -30d and +48hrs of PMN index

See [dx codes](./data/excluded_dx_codes.csv) and [procs](./data/excluded_proc_codes.csv) for list of excluded diagnosis and procedure codes.

In [12]:
# confirm min age >= 18
q = """
select
    (anchor_age + (extract(year from charttime) - anchor_year)) age
from sbp.pmn_index
left join mimiciv_hosp.patients
using (subject_id)
"""
print(f"Min age of SBP group is: {pd.read_sql(q,engine).age.min()} years old")

Min age of SBP group is: 18.0 years old


In [13]:
# exclusion table based on surgical procedures and diagnoses that my result in secondary SBP
pd.read_csv('./data/excluded_dx_codes.csv').to_sql('excluded_dx', schema='sbp', con=engine, if_exists='replace')
pd.read_csv('./data/excluded_proc_codes.csv').to_sql('excluded_procs', schema='sbp', con=engine, if_exists='replace')
q = """
drop table if exists sbp.exclusions;
create table if not exists sbp.exclusions as (
    with dx as (
        select
            dx.subject_id,
            dx.hadm_id,
            admittime,
            icd_code
        from mimiciv_hosp.diagnoses_icd dx
        left join mimiciv_hosp.admissions ad
        on dx.subject_id = ad.subject_id and dx.hadm_id = ad.hadm_id
        and dx.subject_id in (select distinct subject_id from sbp.pmn_index)
    )
    select
        idx.subject_id,
        idx.hadm_id,
        max(case
            when dx.icd_code in (select distinct icd_code from sbp.excluded_dx) then 1
            when proc.icd_code in (select distinct icd_code from sbp.excluded_procs) then 1
            when (pt.anchor_age + (extract(year from idx.charttime) - pt.anchor_year)) < 18 then 1
            else 0
            end
        ) as exclusion
    from sbp.pmn_index idx
    left join dx
    on idx.subject_id = dx.subject_id
    and dx.admittime between (idx.charttime - interval '30 days') and (idx.charttime + interval '48 hours')
    left join mimiciv_hosp.procedures_icd proc
    on idx.subject_id = proc.subject_id
    and proc.chartdate between (idx.charttime - interval '30 days') and (idx.charttime + interval '48 hours')
    left join mimiciv_hosp.patients pt
    on idx.subject_id = pt.subject_id
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Cultures
Cultures obtained +/- 24hrs from SBP index.

In [14]:
# get peritoneal cultures for each patient in PMN index table
q = """
drop table if exists sbp.cx;
create table if not exists sbp.cx as (
    with tmp as (
        select
            me.subject_id,
            me.hadm_id,
            me.micro_specimen_id,
            coalesce(me.charttime, me.chartdate::timestamp) as charttime,
            me.org_name,
            me.ab_name,
            me.interpretation,
            me.spec_type_desc,
            me.test_name,
            max(case when dl.fluid = 'Ascites' then 1 else 0 end) as corresponds_with_ascites_draw
        from sbp.pmn_index idx
        left join mimiciv_hosp.microbiologyevents as me
        on idx.subject_id = me.subject_id
        and idx.hadm_id = me.hadm_id
        left join mimiciv_hosp.labevents as le
        on me.subject_id = le.subject_id
        and me.charttime between (le.charttime - interval '3 hours') and (le.charttime + interval '3 hours')
        left join mimiciv_hosp.d_labitems dl
        using (itemid)
        where spec_type_desc in (
            'FLUID RECEIVED IN BLOOD CULTURE BOTTLES', 'FLUID,OTHER', 'PERITONEAL FLUID'
        )
        and test_name in ('FLUID CULTURE', 'Fluid Culture in Bottles', 'ANAEROBIC CULTURE')
        group by me.subject_id, me.hadm_id, me.micro_specimen_id, me.charttime, me.chartdate, me.org_name, me.ab_name, me.interpretation, me.spec_type_desc, me.test_name        
    ), imputed as (
        -- impute AST data from previous (up to 7 days) for same org if AST missing
        select
            a.subject_id,
            a.hadm_id,
            a.micro_specimen_id,
            a.charttime,
            a.org_name,
            coalesce(a.ab_name, b.ab_name) as ab_name,
            coalesce(a.interpretation, b.interpretation) as interpretation,
            a.spec_type_desc,
            a.test_name,
            a.corresponds_with_ascites_draw
        from tmp a
        left join tmp b
        on a.subject_id = b.subject_id
        and a.org_name = b.org_name
        and b.charttime between (a.charttime - interval '7 days') and (a.charttime)
        and a.ab_name is null
        and b.ab_name is not null 
    )
    select
        *
    from imputed
    where corresponds_with_ascites_draw = 1  -- make sure there was acites draw within +/- 3hrs from culture timej
);   
"""
run_query(q)

In [15]:
# get index cultures +/- 24hrs from PMN
q = """
drop table if exists sbp.cx_index;
create table if not exists sbp.cx_index as
    select
        idx.subject_id,
        idx.hadm_id,
        micro_specimen_id,
        cx.charttime,
        org_name,
        ab_name,
        interpretation,
        spec_type_desc,
        test_name
    from sbp.pmn_index idx
    left join sbp.cx cx
    on idx.subject_id = cx.subject_id
    and idx.hadm_id = cx.hadm_id
    and cx.charttime between (idx.charttime - interval '24 hours') and (idx.charttime + interval '24 hours')
;
"""
run_query(q) 

## MDRO
MDR organisms are defined according to Magiorakos et al. as organisms with acquired non-susceptibility (I/R) to at least one agent in three or more antimicrobial categories.

In [16]:
# create culture table
pd.read_csv('./data/culture_atcc.csv').to_sql('culture_class', schema='sbp', con=engine, if_exists='replace')
q = """
drop table if exists sbp.culture;
create table if not exists sbp.culture as (
    with class_resistance as (
        select
            subject_id,
            hadm_id,
            micro_specimen_id,
            charttime,
            org_name,
            substr(atc,1,5) as atc5,
            max(case when interpretation in ('I','R') then 1 else 0 end) as class_resistant,            
            max(case 
                when org_name like 'STAPH AUREUS%%'
                and ab_name in ('OXACILLIN', 'CEFOXITIN')
                and interpretation in ('I','R')
                then 1 else 0 end
            ) as mrsa,
            max(case
                when org_name in ('ENTEROCOCCUS FAECIUM', 'ENTEROCOCCUS FAECALIS')
                and ab_name = 'VANCOMYCIN' and interpretation in ('I','R')
                then 1 else 0 end
            ) as vre,
            max(case
                when (org_name like 'ENTEROBACTER%%' or org_name in ('ESCHERICHIA COLI', 'KLEBSIELLA PNEUMONIAE'))
                and ab_name in ('MEROPENEM', 'DORIPENEM', 'IMIPENEM', 'ERTAPENEM')
                and interpretation in ('I','R')
                then 1 else 0 end
            ) as cre,
            max(case
                when org_name in ('ESCHERICHIA COLI', 'KLEBSIELLA PNEUMONIAE')
                and ab_name in ('CEFTRIAXONE','CEFOTAXIME' )
                and interpretation in ('I','R')
                then 1 else 0 end
            ) as esbl,
            max(case
                when org_name in (
                    'CITROBACTER FREUNDII COMPLEX',
                    'ENTEROBACTER AEROGENES',
                    'ENTEROBACTER CLOACAE COMPLEX',
                    'ESCHERICHIA COLI',
                    'KLEBSIELLA (RAOULTELLA) ORNITHINOLYTICA',
                    'KLEBSIELLA OXYTOCA',
                    'KLEBSIELLA PNEUMONIAE',
                    'SERRATIA MARCESCENS',
                    'PROTEUS MIRABILIS',
                    'MORGANELLA MORGANII'
                ) and ab_name in (
                    'LEVOFLOXACIN',
                    'CIPROFLOXACIN'
                ) and interpretation in ('I','R')
                then 1 else 0 end
            ) as qrgn                
        from sbp.cx_index
        left join sbp.culture_class
        using (ab_name)
        group by subject_id, hadm_id, micro_specimen_id, charttime, org_name, substr(atc, 1, 5)
    ), flagged_mdrs as (
        select
            subject_id,
            hadm_id,
            micro_specimen_id,
            charttime,
            org_name,
            case when sum(class_resistant) >= 3 then 1 else 0 end as mdr,
            max(mrsa) as mrsa,
            max(vre) as vre,
            max(cre) as cre,
            max(esbl) as esbl,
            max(qrgn) as qrgn
        from class_resistance
        group by subject_id, hadm_id, micro_specimen_id, charttime, org_name
    )
    select
        subject_id,
        hadm_id,
        max(case when mdr.micro_specimen_id is not null then 1 else 0 end) as culture_done,
        max(case when org_name is not null then 1 else 0 end) as culture_positive,
        case
            when sum(case when org_name is not null then 1 else 0 end) > 1 then 1
            else 0
        end as polymicrobial,
        string_agg(org_name || case when mdr = 1 then ' (mdr)' else '' end, '; ') as org_name,
        max(case when mdr = 1 then 1 else 0 end) as mdr,
        max(case when mrsa = 1 then 1 else 0 end) as mrsa,
        max(case when vre = 1 then 1 else 0 end) as vre,
        max(case when cre = 1 then 1 else 0 end) as cre,
        max(case when esbl = 1 then 1 else 0 end) as esbl,
        max(case when qrgn = 1 then 1 else 0 end) as qrgn
    from sbp.pmn_index
    left join flagged_mdrs mdr
    using (subject_id, hadm_id)
    group by subject_id, hadm_id
);
"""
run_query(q)

## Treatment Failure
Treatment failure is defined as either one of the following
- **Non-responder**: Failure to reach PMN < 250 (or 25% reduction) within 48hr to 7 days of SBP
- **Antibiotic coverage** - organism not covered by empiric antibiotic

In [17]:
# create non-responder table
q = """
drop table if exists sbp.non_responder;
create table if not exists sbp.non_responder as (
    with tmp as (
        select
            idx.subject_id,
            idx.hadm_id,
            min(case when fu.pmn_count >= 250 then 1 else 0 end) as tx_failure_pmn_le_250, 
            min(case when fu.pmn_count/idx.pmn_count >= (1 - 0.25) then 1 else 0 end) as tx_failure_pmn_ge_25pct
        from sbp.pmn_index idx
        left join sbp.pmn fu
        on idx.hadm_id = fu.hadm_id
        and fu.charttime between (idx.charttime + interval '48 hours') and (idx.charttime + interval '7 days')
        group by idx.subject_id, idx.hadm_id
    )
    select
        *,
        case when tx_failure_pmn_le_250 = 1 or tx_failure_pmn_ge_25pct = 1 then 1 else 0 end as non_responder
    from tmp
);
"""
run_query(q)

In [18]:
# use ndc-to-atc mapping to create a final drug to atc map table
pd.read_csv('./data/ndc_map.csv', dtype={'ndc':str}).to_sql('ndc_map', schema='sbp', con=engine, index=False, if_exists='replace')
q = """
drop table if exists sbp.drug_atc;
create table if not exists sbp.drug_atc as (
    with tmp as (
        select distinct
            drug,
            atc
        from mimiciv_hosp.prescriptions
        inner join sbp.pmn_index
        using (subject_id)
        left join sbp.ndc_map
        using (ndc)
    )
    select distinct
        a.drug,
        coalesce(a.atc, b.atc) as atc
    from tmp as a
    left join tmp b
    on lower(a.drug) = lower(b.drug)
    and a.atc is null
    and b.atc is not null
    where coalesce(a.atc, b.atc) is not null
);
"""
run_query(q)

In [ ]:
# create emar table for all drugs
q = """
drop table if exists sbp.emar;
create table if not exists sbp.emar as (
    select
        em.subject_id,
        em.hadm_id,
        em.poe_id,
        em.charttime,
        em.medication,
        atc.atc,
        rx.route
    from sbp.pmn_index idx
    left join mimiciv_hosp.emar as em
    on em.subject_id = idx.subject_id
    and event_txt like 'Administered%%'
    left join sbp.drug_atc
    on drug = medication
    left join sbp.drug_atc atc
    on em.medication = atc.drug
    left join mimiciv_hosp.prescriptions rx    
    using (poe_id)
    where route in ('IV','PO/NG','PO','NG','IP')
);
"""
run_query(q)

## Empiric antibiotics
Table of antibiotics given within +/- 48 hours of culture

In [ ]:
# treatment abx classes
q = """
drop table if exists sbp.treatment;
create table if not exists sbp.treatment as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case when medication in ('Amikacin') then 1 else 0 end) as amikacin,
        max(case when medication in ('Amoxicillin') then 1 else 0 end) as amoxicillin,
        max(case when medication in ('Amoxicillin-Clavulanic Acid') then 1 else 0 end) as amox_clav,
        max(case when medication in ('Ampicillin') then 1 else 0 end) as ampicillin,
        max(case when medication in ('Ampicillin-Sulbactam') then 1 else 0 end) as amp_sulbactam,
        max(case when medication in ('Azithromycin') then 1 else 0 end) as azithromycin,
        max(case when medication in ('Aztreonam') then 1 else 0 end) as aztreonam,
        max(case when medication in ('CeFAZolin','CefazoLIN') then 1 else 0 end) as cefazolin,
        max(case when medication in ('CefePIME') then 1 else 0 end) as cefepime,
        max(case when medication in ('Cefpodoxime Proxetil') then 1 else 0 end) as cefpodoxime,
        max(case when medication in ('Ceftaroline') then 1 else 0 end) as ceftaroline,
        max(case when medication in ('CefTAZidime') then 1 else 0 end) as ceftazidime,
        max(case when medication in ('CefTAZidime-Avibactam (Avycaz)') then 1 else 0 end) as avycaz,
        max(case when medication in ('CefTRIAXone','CeftriaXONE') then 1 else 0 end) as ceftriaxone,
        max(case when medication in ('Cephalexin') then 1 else 0 end) as cephalexin,
        max(case when medication in ('Ciprofloxacin HCl') then 1 else 0 end) as cirpofloxacin,
        max(case when medication in ('Clarithromycin') then 1 else 0 end) as clarithromycin,
        max(case when medication in ('Colistin') then 1 else 0 end) as colistin,
        max(case when medication in ('Daptomycin', 'DAPTOmycin') then 1 else 0 end) as dapto,
        max(case when medication in ('Doxycycline Hyclate') then 1 else 0 end) as doxy,
        max(case when medication in ('Ertapenem Sodium') then 1 else 0 end) as ertapenem,
        max(case when medication in ('Erythromycin') then 1 else 0 end) as erythromycin,
        max(case when medication in ('Gentamicin') then 1 else 0 end) as gentamicin,
        max(case when medication in ('Levofloxacin', 'LevoFLOXacin') then 1 else 0 end) as levofloxacin,
        max(case when medication in ('Linezolid') then 1 else 0 end) as linezolid,
        max(case when medication in ('Meropenem') then 1 else 0 end) as meropenem,
        max(case when medication in ('MetroNIDAZOLE', 'MetRONIDAZOLE (FLagyl)') then 1 else 0 end) as metronidazole,
        max(case when medication in ('Minocycline') then 1 else 0 end) as minocycline,
        max(case when medication in ('Nafcillin') then 1 else 0 end) as nafcillin,
        max(case when medication in ('omadacycline') then 1 else 0 end) as omadacycline,
        max(case when medication in ('Piperacillin-Tazobactam') then 1 else 0 end) as pip_tazo,
        max(case when medication in ('Sulfameth/Trimethoprim DS', 'Sulfameth/Trimethoprim SS','Sulfameth/Trimethoprim Suspension','Sulfamethoxazole-Trimethoprim') then 1 else 0 end) as tmp_smx,
        max(case when medication in ('Tigecycline') then 1 else 0 end) as tigecycline,
        max(case when medication in ('Vancomycin') then 1 else 0 end) as vancomycin,
        max(case when medication in ('Fluconazole') then 1 else 0 end) as fluconazole,
        max(case when medication in ('Micafungin') then 1 else 0 end) as micafungin,
        max(case when medication in ('Ambisome') then 1 else 0 end) as ambisome,
        max(case when medication in ('Posaconazole') then 1 else 0 end) as posaconazole        
    from sbp.pmn_index idx
    left join sbp.emar em
    on idx.subject_id = em.subject_id
    and em.charttime between (idx.charttime - interval '48 hours') and (idx.charttime + interval '48 hours')
    and (em.atc like 'J01%%' or em.atc like 'J02%%')    
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

In [ ]:
pd.read_csv('./data/abx_coverage_reviewed.csv').to_sql('abx_coverage_reviewed', schema='sbp', if_exists='replace', index=False, con=engine)
# create treatment failure table
q = """
drop table if exists sbp.treatment_failure;
create table if not exists sbp.treatment_failure as (
    with tmp as (
       select distinct
           cx.subject_id,
           cx.hadm_id,
           cx.micro_specimen_id,
           cx.org_name,
           string_agg( distinct ab_name || ' (' || interpretation || ')', '; ') as culture_result,
           string_agg(distinct medication, '; ') as treatment,
           max(case
               when 
                (
                   (lower(ab_name) = lower(medication))
                   or
                   (ab_name = 'PIPERACILLIN/TAZO' and medication = 'Piperacillin-Tazobactam')
                   or
                   (ab_name = 'CIPROFLOXACIN' and medication = 'Ciprofloxacin HCl')
                ) and interpretation not in ('I','R') then 1
                else 0
           end) as treatment_covers,
           min(case when medication is null then 1 else 0 end) as no_abx_given
       from sbp.cx_index cx
       left join sbp.emar em
       on cx.subject_id = em.subject_id
       and em.charttime between (cx.charttime - interval '48 hours') and (cx.charttime + interval '48 hours')
       and (em.atc like 'J01%%' or em.atc like 'J02%%')
       where cx.org_name is not null
       group by cx.subject_id, cx.hadm_id, cx.micro_specimen_id, cx.org_name
    ), final_covered as (
        select
            subject_id,
            hadm_id,
            micro_specimen_id,
            org_name,
            culture_result,
            treatment,
            coalesce(
                covered,   -- manually reviewed determiniation
                case 
                    when treatment_covers = 1 then 1
                    when no_abx_given = 1 then 0
                else 0 end
            ) as final_covered,
            case
                when covered = 1 then 1
                when covered = 0 then 0
                else null
            end as manually_reviewed_coverage,
            case
                when treatment_covers = 1 then 1
                when no_abx_given = 1 then 0
                else null
            end as inferred_coverage
        from tmp
        left join sbp.abx_coverage_reviewed
        using (subject_id, hadm_id, micro_specimen_id, org_name)
    ), rollup as (
        select
            subject_id,
            hadm_id,
            string_agg(coalesce(org_name, 'NA') || '|' || coalesce(culture_result, 'NA') || '|' || coalesce(treatment, 'NA'), ';') as org_names,
            min(final_covered) as covered
        from final_covered
        group by subject_id, hadm_id        
    ), failure as (
        select
            subject_id,
            hadm_id,
            org_names,
            case
                when covered is null then 0      -- not a treatment failure if no bug (?)
                when covered = 1 then 0          -- not a treatment failure if bug is covered
                else 1                           -- otherwise (e.g covered = 0) treatment failure
            end as treatment_failure,
            non_responder
        from sbp.pmn_index
        left join rollup
        using (subject_id, hadm_id)
        left join sbp.non_responder
        using (subject_id, hadm_id)
    )
    select
        subject_id,
        hadm_id,
        treatment_failure,
        non_responder,
        case when treatment_failure = 1 or non_responder = 1 then 1 else 0 end as composite_failure
    from failure            
);
"""
run_query(q)

## Prior antibiotic exposure
Antibiotic exposure as determined from EMAR over prior 7, 14, 30, 90, 365 days from SBP index.

In [ ]:
# prior abx/antifungal exposure
q = """
drop table if exists sbp.prior_abx_exposure;
create table if not exists sbp.prior_abx_exposure as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case when (idx.charttime - emar.charttime) <= interval '7 days' then 1 else 0 end) as abx_expo_le_7d,
        max(case when (idx.charttime - emar.charttime) <= interval '14 days' then 1 else 0 end) as abx_expo_le_14d,
        max(case when (idx.charttime - emar.charttime) <= interval '30 days' then 1 else 0 end) as abx_expo_le_30d,
        max(case when (idx.charttime - emar.charttime) <= interval '90 days' then 1 else 0 end) as abx_expo_le_90d,
        max(case when (idx.charttime - emar.charttime) <= interval '365 days' then 1 else 0 end) as abx_expo_le_365d
    from sbp.pmn_index idx
    left join sbp.emar emar
    on idx.subject_id = emar.subject_id
    and emar.charttime < idx.charttime
    and idx.hadm_id != emar.hadm_id
    and (emar.atc like 'J01%%' or emar.atc like 'J02%%')
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Prior PPI exposure
PPI exopsure as determined from EMAR over prior 7, 14, 30, 90, 365 days from SBP index.

In [ ]:
# prior ppi exposure
q = """
drop table if exists sbp.ppi;
create table if not exists sbp.ppi as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case when (idx.charttime - rx.starttime) <= interval '7 days' then 1 else 0 end) as ppi_expo_le_7d,
        max(case when (idx.charttime - rx.starttime) <= interval '14 days' then 1 else 0 end) as ppi_expo_le_14d,
        max(case when (idx.charttime - rx.starttime) <= interval '30 days' then 1 else 0 end) as ppi_expo_le_30d,
        max(case when (idx.charttime - rx.starttime) <= interval '90 days' then 1 else 0 end) as ppi_expo_le_90d,
        max(case when (idx.charttime - rx.starttime) <= interval '365 days' then 1 else 0 end) as ppi_expo_le_365d
    from sbp.pmn_index idx
    left join mimiciv_hosp.prescriptions rx
    on idx.subject_id = rx.subject_id
    and rx.starttime < idx.charttime
    and rx.hadm_id != idx.hadm_id
    left join sbp.drug_atc d
    on rx.drug = d.drug
    and d.atc like 'A02BC%%'
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Baseline vasopressors
Whether or not vasopressors were administered within +/- 24hrs from SBP index

In [ ]:
# vasopressors at baseline
q = """
drop table if exists sbp.pressors;
create table if not exists sbp.pressors as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case when va.stay_id is not null then 1 else 0 end) as baseline_pressors
    from sbp.pmn_index as idx
    left join mimiciv_icu.icustays icu
    using (subject_id, hadm_id)
    left join mimiciv_derived.vasoactive_agent va
    on icu.stay_id = va.stay_id
    and (va.starttime, va.endtime) overlaps ((idx.charttime - interval '24 hours'), (idx.charttime + interval '24 hours'))
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Prior hospital exposure
Hospital exopsure within previous 7,14,30,90,365 days from SBP index.

In [ ]:
# prior hospital exposure
q = """
drop table if exists sbp.prior_hx_exposure;
create table if not exists sbp.prior_hx_exposure as (
    select
        idx.subject_id,
        idx.hadm_id, 
        max(case when (idx.charttime - adm.dischtime) <= interval '7 days' then 1 else 0 end) as hosp_expo_le_7d,
        max(case when (idx.charttime - adm.dischtime) <= interval '14 days' then 1 else 0 end) as hosp_expo_le_14d,
        max(case when (idx.charttime - adm.dischtime) <= interval '30 days' then 1 else 0 end) as hosp_expo_le_30d,
        max(case when (idx.charttime - adm.dischtime) <= interval '90 days' then 1 else 0 end) as hosp_expo_le_90d,
        max(case when (idx.charttime - adm.dischtime) <= interval '365 days' then 1 else 0 end) as hosp_expo_le_365d
    from sbp.pmn_index idx
    left join mimiciv_hosp.admissions adm
    on idx.subject_id = adm.subject_id
    and adm.dischtime < idx.charttime
    and adm.hadm_id != idx.hadm_id
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## SBP Recurrence
Any new SBP event occurring after SBP index.

In [ ]:
# sbp recurrence...any new infection after initial had resolved
q = """
drop table if exists sbp.sbp_recurrence;
create table if not exists sbp.sbp_recurrence as (
    with sbp_resolved as (
        select
            idx.subject_id,
            idx.hadm_id,
            max(case when ep.sbp = 1 then 1 else 0 end) as sbp_recurrence
        from sbp.pmn_index idx
        left join sbp.sbp_episodes ep
        on idx.subject_id = ep.subject_id
        and ep.starttime between (idx.charttime + interval '1 days') and (idx.charttime + interval '30 days')
        group by idx.subject_id, idx.hadm_id
    )
    select * from sbp_resolved
);
"""
run_query(q)

## AKI
Defined as SCr increase >= 0.3 mg/dL within 48hrs OR 1.5 fold increase from baseline within 7 day period.

Indicates presence of resolved AKI (stoptime prior to PMN index), ongoing AKI (PMN index between starttime, stoptime), incident AKI (starttime after PMN index)

In [ ]:
# serum creatinine table for all patients in PMN index
q = """
drop table if exists sbp.scr;
create table if not exists sbp.scr as (
    select
        idx.subject_id,
        idx.hadm_id,
        le.charttime,
        valuenum as scr
    from sbp.pmn_index idx
    left join mimiciv_hosp.labevents le
    on idx.subject_id = le.subject_id
    and idx.hadm_id = le.hadm_id
    and le.itemid = 50912
    where valuenum is not null
);
"""
run_query(q)

In [ ]:
# create table of AKI episodes
q = """
drop table if exists sbp.aki_episodes;
create table if not exists sbp.aki_episodes as (
    with flagged_aki_events as (
        select
            idx.subject_id,
            idx.hadm_id,
            idx.charttime,
            idx.scr as scr,
            case
                when (idx.scr - min(cr1.scr)) >= 0.3 then 1
                when (idx.scr/min(cr2.scr)) >= 1.5 then 1
                else 0
            end as aki_event
        from sbp.scr idx
        left join sbp.scr cr1
        on idx.subject_id = cr1.subject_id
        and cr1.charttime between (idx.charttime - interval '48 hours') and (idx.charttime + interval '48 hours')
        left join sbp.scr cr2
        on idx.subject_id = cr2.subject_id
        and cr2.charttime between (idx.charttime - interval '7 days') and (idx.charttime + interval '7 days')
        group by idx.subject_id, idx.hadm_id, idx.charttime, idx.scr    
    ), aki_events as (
        select
            *
        from flagged_aki_events
        where aki_event = 1
    ), lags as (
        select
            *,
            lag(charttime) over (partition by subject_id, hadm_id order by charttime) as prev_charttime
        from aki_events
    ), new_episodes as (
        select
            subject_id,
            hadm_id,
            charttime,
            scr,
            case
                when prev_charttime is null then 1
                when (charttime - prev_charttime) >= interval '7 days' then 1
                else 0
            end as new_episode
        from lags
    ), flagged_episodes as (
        select
            subject_id,
            hadm_id,
            charttime,
            scr,
            1 - new_episode as part_of_previous_episode,
            sum(new_episode) over (
                partition by
                    subject_id,
                    hadm_id
                order by charttime
                rows between unbounded preceding and current row
            ) as episode_id
        from new_episodes
    ), start_stop_times as (
        select
            subject_id,
            hadm_id,
            episode_id,
            charttime,
            case when charttime = min(charttime) over (partition by subject_id, hadm_id, episode_id) then scr else null end as start_scr,
            case when charttime = max(charttime) over (partition by subject_id, hadm_id, episode_id) then scr else null end as stop_scr
        from flagged_episodes
    ), episodes as (
        select
            subject_id,
            hadm_id,
            episode_id,
            1 as aki,
            min(charttime) as starttime,
            max(charttime) as stoptime,    
            max(start_scr) as start_scr,
            max(stop_scr) as stop_scr
        from start_stop_times
        group by subject_id, hadm_id, episode_id
    )
    select * from episodes
);
"""
run_query(q)

In [ ]:
# create table of AKI events in relation to SBP index
q = """
drop table if exists sbp.aki;
create table if not exists sbp.aki as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case when aki.aki = 1 then 1 else 0 end) as any_aki,
        max(case when aki.stoptime < idx.charttime then 1 else 0 end) as resolved_aki,
        max(case when idx.charttime between aki.starttime and aki.stoptime then 1 else 0 end) as ongoing_aki,
        max(case when aki.starttime >= idx.charttime then 1 else 0 end) as indicent_aki 
    from sbp.pmn_index idx
    left join sbp.aki_episodes aki
    on idx.subject_id = aki.subject_id
    and idx.hadm_id = aki.hadm_id
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Meld score

In [ ]:
q = """
drop table if exists sbp.meld;
create table if not exists sbp.meld as (
    select
        subject_id,
        hadm_id,
        max(meld) as meld
    from sbp.pmn_index
    left join mimiciv_derived.meld
    using (subject_id, hadm_id)
    group by subject_id, hadm_id
);
"""

run_query(q)
        

## History of ESRD/CKD
Any history of ESRD/CKD diagnosis codes or administered RRT prior to SBP index visit.

In [ ]:
# hx of esrd
q = """
drop table if exists sbp.esrd;
create table if not exists sbp.esrd as (
    select
        idx.subject_id,
        idx.hadm_id,
        max(case
            -- history of CKD/ESRD diagnosis
            when dx.icd_code like 'N18%%' then 1                    -- CKD
            when dx.icd_code in ('Z992', '5856', 'V4511') then 1    -- dependent on renal dialysis, ESRD, renal dialysis status
            when rrt.dialysis_present = 1 then 1 -- prior history of dialysis
            else 0
            end
        ) as esrd_hx
    from sbp.pmn_index idx
    left join (
        select
            *
        from mimiciv_hosp.diagnoses_icd
        inner join mimiciv_hosp.admissions 
        using (subject_id, hadm_id)
    ) as dx
    on idx.subject_id = dx.subject_id
    and dx.admittime < idx.charttime
    and idx.hadm_id != dx.hadm_id
    left join (
        select
            subject_id,
            hadm_id,
            rrt.*
        from mimiciv_derived.rrt as rrt
        left join mimiciv_icu.icustays
        using (stay_id)
    ) as rrt
    on idx.subject_id = rrt.subject_id
    and idx.hadm_id != rrt.hadm_id
    and rrt.charttime < idx.charttime
    group by idx.subject_id, idx.hadm_id
);
"""
run_query(q)

## Hospital LOS
Length of stay in days for SBP index admission.

In [ ]:
# length of hospital stay
q = """
drop table if exists sbp.los;
create table if not exists sbp.los as (
    select
        subject_id,
        hadm_id,
        extract(epoch from (dischtime - admittime))/(60*60*24) as los_days
    from sbp.pmn_index
    left join mimiciv_hosp.admissions
    using (subject_id, hadm_id)
);
"""
run_query(q)

## Mortality
In-hospital mortality and death within 30,90,365 days from SBP index admission.

In [ ]:
# mortality...in hospital, and all-cause at 30, 90, and 365 days
q = """
drop table if exists sbp.mortality;
create table if not exists sbp.mortality as (
    select
        subject_id,
        hadm_id,
        hospital_expire_flag as death_at_visit,
        case when hospital_expire_flag = 1 or dod between a.charttime and a.charttime + interval '7 days' then 1 else 0 end as death_le_7d,
        case when hospital_expire_flag = 1 or dod between a.charttime and a.charttime + interval '14 days' then 1 else 0 end as death_le_14d,
        case when hospital_expire_flag = 1 or dod between a.charttime and a.charttime + interval '30 days' then 1 else 0 end as death_le_30d,
        case when hospital_expire_flag = 1 or dod between a.charttime and a.charttime + interval '90 days' then 1 else 0 end as death_le_90d,
        case when hospital_expire_flag = 1 or dod between a.charttime and a.charttime + interval '365 days' then 1 else 0 end as death_le_365d
    from sbp.pmn_index a
    left join mimiciv_hosp.admissions
    using (subject_id, hadm_id)
    left join mimiciv_hosp.patients
    using (subject_id)
);
"""
run_query(q)

## Final table
Compile the final table

In [ ]:
q = """
drop table if exists sbp.final;
create table if not exists sbp.final as (
    select
        *
    from sbp.pmn_index
    left join sbp.demo using (subject_id, hadm_id)
    left join sbp.community_acquired using (subject_id, hadm_id)
    left join sbp.exclusions using (subject_id, hadm_id)
    left join sbp.meld using (subject_id, hadm_id)
    left join sbp.esrd using (subject_id, hadm_id)
    left join sbp.aki using (subject_id, hadm_id)
    left join sbp.los using (subject_id, hadm_id)
    left join sbp.culture using (subject_id, hadm_id)
    left join sbp.treatment using (subject_id, hadm_id)
    left join sbp.mortality using (subject_id, hadm_id)
    left join sbp.pmn_dx using (subject_id, hadm_id)
    left join sbp.treatment_failure using (subject_id, hadm_id)
    left join sbp.ppi using (subject_id, hadm_id)
    left join sbp.pressors using (subject_id, hadm_id)
    left join sbp.prior_abx_exposure using (subject_id, hadm_id)
    left join sbp.prior_hx_exposure using (subject_id, hadm_id)
    left join sbp.sbp_recurrence using (subject_id, hadm_id)
);
"""
run_query(q)

In [ ]:
tmp = query_table('sbp.demo')

# Analysis

In [ ]:
from tableone import TableOne
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_sql('select * from sbp.final', engine, dtype={'subject_id':'Int64', 'hadm_id':'Int64'})
print(f"Initial count: {df.shape[0]}")
df = df.query('exclusion == 0')
print(f"After exclusions: {df.shape[0]}")

In [ ]:
def generate_table(df: "DataFrame", groupby: str, path: str):
    tmp = df.drop(columns=['subject_id','hadm_id', 'charttime', 'org_name']) 
    cat = tmp.select_dtypes('int64').columns.tolist()
    cont = tmp.select_dtypes('float64').columns.tolist()
    nom = tmp.select_dtypes('str').columns.tolist()
    tmp[cat] = tmp[cat].replace({0:'No',1:'Yes'})
    table = TableOne(
        tmp,
        categorical=cat + nom,
        continuous=cont,
        groupby=groupby,
        pval=True if groupby else None,
        missing=False,
        normal_test=True,
        limit={c: (1 if c in cat else 10) for c in cat + nom},
        order={c: ['Yes', 'No'] for c in cat},
        nonnormal=['los_days','pmn_count'],
    )
    with open(path, 'w') as f:
        f.write(table.tabulate(tablefmt="github"))

In [ ]:
overall = generate_table(df, None, './tables/overall.md')
all_by_composite_failure = generate_table(df, 'composite_failure', './tables/all-by-composite-failure.md')
all_by_non_responder = generate_table(df, 'non_responder', './tables/all-by-non-responder.md')
all_by_treatment_failure = generate_table(df, 'treatment_failure', './tables/all-by-treatment-failure.md')
all_by_culture_done = generate_table(df, 'culture_done', './tables/all-by-culture-done.md')

In [ ]:
cult_pos = df.query('culture_positive == 1')
cult_pos_by_mdr = generate_table(cult_pos, 'mdr', './tables/culture-positive-by-mdr.md')
cult_pos_by_composite_failure = generate_table(cult_pos, 'composite_failure', './tables/culture-positive-by-composite-failure.md')
cult_pos_by_non_responder = generate_table(cult_pos, 'non_responder', './tables/culture-positive-by-non-responder.md')
cult_pos_by_treatment_failure = generate_table(cult_pos, 'treatment_failure', './tables/culture-positive-by-treatment-failure.md')

In [ ]:
no_esrd = df.query('esrd_hx == 0')
no_esrd_by_mdr = generate_table(no_esrd,'mdr', './tables/no-esrd-by-mdr.md')